# Module 04: Fine-Tuning Techniques

**Companion notebook for**: `04-fine-tuning.html`

## Learning Objectives
- Understand when to fine-tune vs. when to use prompt engineering or RAG
- Learn the LoRA and QLoRA parameter-efficient fine-tuning techniques and calculate their memory savings
- Prepare training datasets in JSONL format for OpenAI fine-tuning
- Walk through the full OpenAI fine-tuning API lifecycle: upload, train, monitor, and use a fine-tuned model
- Implement training data quality checks (validation, deduplication, format enforcement)
- Select hyperparameters and estimate fine-tuning costs
- Evaluate fine-tuned models against base model baselines

## Prerequisites
- OpenAI API key set as `OPENAI_API_KEY` environment variable
- Basic understanding of LLMs and the Chat Completions API (Module 03)

---
## Setup & Dependencies

In [ ]:
%pip install -q openai tiktoken

In [ ]:
import os
import json
import time
import hashlib
import random
from collections import Counter

import tiktoken
from openai import OpenAI

# Verify API key is available
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY environment variable"

client = OpenAI()  # reads OPENAI_API_KEY from env
print("All imports successful. OpenAI API key found.")

---
## Section 1: Fine-Tuning vs Prompting vs RAG -- When to Use Which

The most important decision in any fine-tuning project is whether you need fine-tuning at all.

**The diagnostic question**: Is this a **knowledge problem** or a **behavior problem**?

| Problem Type | Description | Best Solution |
|---|---|---|
| **Knowledge** | Model lacks specific facts (your product catalog, regulatory updates, internal docs) | **RAG** -- inject facts at query time |
| **Behavior** | Model knows the info but uses wrong tone, format, vocabulary, or style | **Fine-tuning** -- bake behavior into weights |
| **Direction** | Model can do it but needs guidance | **Prompt engineering** -- free, fast, reversible |

**Rule of thumb**: Start with prompt engineering. If you hit a ceiling, try RAG. If you still have a persistent behavior problem, then consider fine-tuning. The three approaches are not mutually exclusive -- production systems often use all three together.

### When fine-tuning wins
1. **Consistent output format** -- structured JSON with specific schemas, produced without format instructions
2. **Specialized vocabulary** -- medical, legal, financial domain fluency at the register level
3. **Latency and cost at scale** -- a fine-tuned 7B model can match prompted GPT-4 on narrow tasks at 10-100x lower cost

In [ ]:
# Decision framework: Should you fine-tune?
# Walk through this checklist before committing to a fine-tuning project.

def should_fine_tune(answers: dict) -> str:
    """
    Simple decision framework for whether to fine-tune.
    
    Args:
        answers: dict with boolean answers to diagnostic questions
    
    Returns:
        Recommendation string
    """
    recommendations = []

    if not answers.get("tried_prompt_engineering", False):
        recommendations.append(
            "START HERE -> Try prompt engineering first. It is free, fast, and reversible."
        )

    if answers.get("knowledge_gap", False):
        recommendations.append(
            "KNOWLEDGE GAP -> Use RAG instead. Fine-tuning is poor at injecting new facts."
        )

    if answers.get("data_changes_frequently", False):
        recommendations.append(
            "DYNAMIC DATA -> Use RAG. Fine-tuned models freeze to their training distribution."
        )

    if answers.get("behavior_problem", False) and answers.get("tried_prompt_engineering", False):
        recommendations.append(
            "BEHAVIOR PROBLEM + PROMPTING EXHAUSTED -> Fine-tuning is a good fit."
        )

    if answers.get("high_call_volume", False):
        recommendations.append(
            "HIGH VOLUME -> Fine-tuning amortizes well. Removing 500 tokens of system "
            "prompt from 10M calls/month saves 5B input tokens/month."
        )

    if answers.get("needs_consistent_format", False):
        recommendations.append(
            "FORMAT CONSISTENCY -> Fine-tuning excels here. Burns format into weights."
        )

    if not recommendations:
        recommendations.append("No strong signal either way. Start with prompt engineering.")

    return "\n".join(f"  {i+1}. {r}" for i, r in enumerate(recommendations))


# Example: Medical note summarization use case
print("Use Case: Medical note summarization with specific JSON output\n")
result = should_fine_tune({
    "tried_prompt_engineering": True,
    "knowledge_gap": False,
    "data_changes_frequently": False,
    "behavior_problem": True,
    "high_call_volume": True,
    "needs_consistent_format": True,
})
print(result)

print("\n" + "="*60)
print("\nUse Case: Answering questions about latest product catalog\n")
result = should_fine_tune({
    "tried_prompt_engineering": True,
    "knowledge_gap": True,
    "data_changes_frequently": True,
    "behavior_problem": False,
    "high_call_volume": False,
    "needs_consistent_format": False,
})
print(result)

---
## Section 2: LoRA & QLoRA -- Parameter-Efficient Fine-Tuning

### LoRA (Low-Rank Adaptation)

Instead of updating all parameters during fine-tuning, LoRA freezes the original weights and introduces small trainable matrices alongside them.

For a weight matrix W0 of shape (d x k), LoRA decomposes the update as:

```
delta_W = B @ A
```

where B has shape (d x r) and A has shape (r x k), with r << min(d, k).

The modified forward pass becomes:
```
h = W0 @ x + (alpha / r) * B @ A @ x
```

**Key hyperparameters**:
- **Rank r** (4, 8, 16, 32, 64): Controls expressiveness. r=8 or r=16 is the sweet spot for most tasks.
- **Alpha**: Scaling factor. Common convention: alpha = r (unscaled) or alpha = 2r.
- **Target modules**: Originally q_proj/v_proj only; modern practice applies to all linear layers.

### QLoRA (Quantized LoRA)

QLoRA combines LoRA with 4-bit NF4 quantization of the base model, reducing memory by ~4x. Three innovations:
1. **NF4 quantization** -- optimal for normally-distributed weights
2. **Double quantization** -- quantizes the quantization constants themselves
3. **Paged AdamW** -- pages optimizer states to CPU RAM on memory spikes

In [ ]:
# LoRA Parameter Calculator
# Understand the memory savings LoRA provides over full fine-tuning.

def lora_parameter_calculator(
    model_params_billions: float,
    hidden_dim: int,
    num_layers: int,
    rank: int = 16,
    alpha: int = 32,
    target_modules: list[str] = None,
) -> dict:
    """
    Calculate LoRA adapter size and memory requirements.

    Args:
        model_params_billions: Total model parameters in billions (e.g. 7.0)
        hidden_dim: Model hidden dimension (e.g. 4096 for Llama-7B)
        num_layers: Number of transformer layers (e.g. 32)
        rank: LoRA rank r
        alpha: LoRA alpha scaling factor
        target_modules: List of module types to apply LoRA to

    Returns:
        Dictionary of parameter counts and memory estimates
    """
    if target_modules is None:
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"]

    total_model_params = int(model_params_billions * 1e9)

    # Attention projections: (hidden_dim x hidden_dim) each
    # MLP projections: gate/up are (hidden_dim x 4*hidden_dim),
    #                  down is (4*hidden_dim x hidden_dim)
    mlp_intermediate = hidden_dim * 4  # approximate

    module_shapes = {
        "q_proj": (hidden_dim, hidden_dim),
        "k_proj": (hidden_dim, hidden_dim),
        "v_proj": (hidden_dim, hidden_dim),
        "o_proj": (hidden_dim, hidden_dim),
        "gate_proj": (hidden_dim, mlp_intermediate),
        "up_proj": (hidden_dim, mlp_intermediate),
        "down_proj": (mlp_intermediate, hidden_dim),
    }

    # LoRA adds B (d x r) + A (r x k) per module per layer
    lora_params_per_layer = 0
    details = []
    for mod in target_modules:
        if mod in module_shapes:
            d, k = module_shapes[mod]
            params = d * rank + rank * k  # B: d*r, A: r*k
            lora_params_per_layer += params
            details.append(f"    {mod}: B({d}x{rank}) + A({rank}x{k}) = {params:,} params")

    total_lora_params = lora_params_per_layer * num_layers
    trainable_pct = (total_lora_params / total_model_params) * 100

    # Memory estimates (bytes)
    base_model_fp16 = total_model_params * 2           # 2 bytes per param in FP16
    base_model_4bit = total_model_params * 0.5         # 0.5 bytes per param in 4-bit
    lora_fp16 = total_lora_params * 2
    optimizer_states = total_lora_params * 8            # Adam: 2 momentum tensors x 4 bytes
    gradients = total_lora_params * 4                   # FP32 gradients

    # Full fine-tuning memory for comparison
    full_ft_memory = total_model_params * 2 + total_model_params * 4 + total_model_params * 8

    gb = 1024**3

    return {
        "model_params": f"{total_model_params:,}",
        "lora_params": f"{total_lora_params:,}",
        "trainable_pct": f"{trainable_pct:.4f}%",
        "rank": rank,
        "alpha": alpha,
        "effective_scaling": f"{alpha/rank:.1f}",
        "target_modules": target_modules,
        "per_layer_detail": details,
        "memory": {
            "full_ft_total_gb": f"{full_ft_memory / gb:.1f} GB",
            "lora_base_fp16_gb": f"{base_model_fp16 / gb:.1f} GB",
            "lora_base_4bit_gb": f"{base_model_4bit / gb:.1f} GB",
            "lora_adapter_mb": f"{lora_fp16 / (1024**2):.1f} MB",
            "lora_optimizer_mb": f"{optimizer_states / (1024**2):.1f} MB",
            "lora_total_fp16_gb": f"{(base_model_fp16 + lora_fp16 + optimizer_states + gradients) / gb:.1f} GB",
            "qlora_total_4bit_gb": f"{(base_model_4bit + lora_fp16 + optimizer_states + gradients) / gb:.1f} GB",
        },
    }


# Example: Llama-3.1-8B
print("=" * 60)
print("LoRA Analysis: Llama-3.1-8B")
print("=" * 60)
result = lora_parameter_calculator(
    model_params_billions=8.0,
    hidden_dim=4096,
    num_layers=32,
    rank=16,
    alpha=32,
)
print(f"\nTotal model params:    {result['model_params']}")
print(f"LoRA adapter params:   {result['lora_params']}")
print(f"Trainable:             {result['trainable_pct']}")
print(f"Rank: {result['rank']}, Alpha: {result['alpha']}, Scaling: {result['effective_scaling']}")
print(f"\nPer-layer breakdown:")
for line in result['per_layer_detail']:
    print(line)
print(f"\nMemory comparison:")
for k, v in result['memory'].items():
    print(f"  {k:30s} {v}")

In [ ]:
# Compare across model sizes and techniques
print(f"{'Model':<20} {'Full FT':>12} {'LoRA (FP16)':>14} {'QLoRA (4-bit)':>15} {'Adapter Size':>14}")
print("-" * 78)

configs = [
    ("Llama-3.1-8B",   8.0,  4096, 32),
    ("Mistral-7B",     7.0,  4096, 32),
    ("Llama-3.1-70B",  70.0, 8192, 80),
    ("Qwen-2.5-72B",   72.0, 8192, 80),
]

for name, params_b, hidden, layers in configs:
    r = lora_parameter_calculator(params_b, hidden, layers, rank=16, alpha=32)
    print(f"{name:<20} {r['memory']['full_ft_total_gb']:>12} "
          f"{r['memory']['lora_total_fp16_gb']:>14} "
          f"{r['memory']['qlora_total_4bit_gb']:>15} "
          f"{r['memory']['lora_adapter_mb']:>14}")

print("\nKey insight: QLoRA makes 70B model fine-tuning feasible on a single A100 80GB GPU.")
print("Without it, 70B full fine-tuning requires ~980 GB -- a cluster of GPUs.")

In [ ]:
# LoRA rank sensitivity: how rank affects adapter size and expressiveness
print("Rank vs Adapter Size (Llama-3.1-8B, all linear layers)\n")
print(f"{'Rank':>6} {'LoRA Params':>14} {'Trainable %':>13} {'Adapter Size':>14}")
print("-" * 50)

for rank in [4, 8, 16, 32, 64, 128]:
    r = lora_parameter_calculator(8.0, 4096, 32, rank=rank, alpha=rank * 2)
    print(f"{rank:>6} {r['lora_params']:>14} {r['trainable_pct']:>13} {r['memory']['lora_adapter_mb']:>14}")

print("\nGuidelines:")
print("  r=4-8:    Simple behavior changes (format, tone)")
print("  r=16:     General-purpose sweet spot")
print("  r=32-64:  Complex domain adaptation")
print("  r=128:    Rarely needed; risk of overfitting on small datasets")

---
## Section 3: Fine-Tuning Data Format Preparation

OpenAI fine-tuning uses JSONL files where each line is a JSON object with a `messages` array. This is the Chat Completions format:

```json
{"messages": [{"role": "system", "content": "..."}, {"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}
```

### Quality > Quantity
100 perfectly curated examples consistently outperform 10,000 mediocre ones for instruction tuning. The model is already highly capable -- it just needs to be steered.

### Dataset size guidelines
| Goal | Examples Needed |
|---|---|
| New output format (always respond as JSON) | 100 -- 500 |
| Communication style / tone shift | 500 -- 2,000 |
| Domain vocabulary & conventions | 2,000 -- 10,000 |
| Substantial new knowledge | 10,000 -- 50,000 |
| Significant new capabilities | 50,000+ |

In [ ]:
# Create a sample fine-tuning dataset for a medical note summarizer.
# In production, these would come from real clinical data or strong-model synthesis.

SYSTEM_PROMPT = (
    "You are a medical documentation assistant. Given a clinical note, produce a "
    "structured JSON summary with fields: chief_complaint, assessment, plan, and icd10_codes."
)

# Sample training examples
training_examples = [
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": (
                    "Patient is a 54-year-old male presenting with substernal chest pain "
                    "radiating to the left arm, onset 2 hours ago. History of hypertension "
                    "and type 2 diabetes. ECG shows ST elevation in leads II, III, aVF. "
                    "Troponin I elevated at 2.4 ng/mL."
                ),
            },
            {
                "role": "assistant",
                "content": json.dumps({
                    "chief_complaint": "Substernal chest pain radiating to left arm, 2h duration",
                    "assessment": "Acute inferior STEMI with positive troponin. High risk given HTN and DM2 history.",
                    "plan": "Emergent cardiac catheterization. Aspirin 325mg, heparin drip, clopidogrel loading. Serial troponins q6h. Cardiology consult stat.",
                    "icd10_codes": ["I21.19", "I10", "E11.9"]
                }),
            },
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": (
                    "72-year-old female with progressive shortness of breath over 3 weeks. "
                    "Bilateral lower extremity edema. JVD present. BNP 1840 pg/mL. "
                    "Chest X-ray shows bilateral pleural effusions and cardiomegaly. "
                    "EF 25% on prior echo."
                ),
            },
            {
                "role": "assistant",
                "content": json.dumps({
                    "chief_complaint": "Progressive dyspnea x3 weeks with bilateral LE edema",
                    "assessment": "Acute on chronic systolic heart failure exacerbation. EF 25%. Volume overloaded with bilateral effusions.",
                    "plan": "IV furosemide 40mg BID. Strict I&O. Daily weights. Fluid restrict 1.5L. Repeat echo. Cardiology consult for optimization.",
                    "icd10_codes": ["I50.22", "J91.8"]
                }),
            },
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": (
                    "45-year-old male presents with sudden onset severe headache described as "
                    "worst headache of my life. Neck stiffness and photophobia. No focal "
                    "neurological deficits. CT head without contrast is negative. "
                    "LP performed showing xanthochromia."
                ),
            },
            {
                "role": "assistant",
                "content": json.dumps({
                    "chief_complaint": "Sudden onset thunderclap headache with meningismus",
                    "assessment": "Subarachnoid hemorrhage confirmed by xanthochromic CSF despite negative CT. High suspicion for ruptured aneurysm.",
                    "plan": "CTA head and neck stat. Neurosurgery consult. Nimodipine 60mg q4h. ICU admission. Strict BP control SBP<140. Seizure precautions.",
                    "icd10_codes": ["I60.9", "G44.1"]
                }),
            },
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": (
                    "28-year-old female with 3-day history of right lower quadrant abdominal "
                    "pain, nausea, and low-grade fever. WBC 14,200. CT abdomen shows "
                    "dilated appendix at 11mm with periappendiceal fat stranding. No perforation."
                ),
            },
            {
                "role": "assistant",
                "content": json.dumps({
                    "chief_complaint": "RLQ abdominal pain x3 days with nausea and low-grade fever",
                    "assessment": "Acute uncomplicated appendicitis. CT-confirmed with dilated appendix and fat stranding. No evidence of perforation.",
                    "plan": "NPO. IV fluids. Surgical consult for laparoscopic appendectomy. Pre-op antibiotics cefoxitin 2g IV. Pain management with IV morphine PRN.",
                    "icd10_codes": ["K35.80"]
                }),
            },
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": (
                    "63-year-old male with type 2 diabetes presenting with fasting glucose of "
                    "280 mg/dL and HbA1c of 10.2%. Currently on metformin 1000mg BID. Reports "
                    "polyuria, polydipsia, and 15-pound weight loss over 2 months. "
                    "Creatinine 1.1, eGFR 72."
                ),
            },
            {
                "role": "assistant",
                "content": json.dumps({
                    "chief_complaint": "Uncontrolled type 2 diabetes with hyperglycemic symptoms and weight loss",
                    "assessment": "Poorly controlled DM2 on metformin monotherapy. HbA1c 10.2% with symptomatic hyperglycemia. Renal function mildly reduced but adequate for metformin.",
                    "plan": "Add semaglutide 0.25mg SC weekly, titrate to 1mg. Continue metformin. Diabetes education referral. Recheck HbA1c in 3 months. Annual eye and foot exam. Lipid panel.",
                    "icd10_codes": ["E11.65", "R63.4"]
                }),
            },
        ]
    },
]

print(f"Created {len(training_examples)} training examples")
print(f"\nSample (first example):")
print(json.dumps(training_examples[0], indent=2)[:500] + "...")

In [ ]:
# Write training and validation JSONL files

def write_jsonl(data: list[dict], filepath: str) -> None:
    """Write a list of dicts to a JSONL file (one JSON object per line)."""
    with open(filepath, "w") as f:
        for example in data:
            f.write(json.dumps(example) + "\n")
    print(f"Wrote {len(data)} examples to {filepath}")


# In a real project you would have hundreds of examples.
# Here we split our small sample to demonstrate the format.
train_data = training_examples[:4]
val_data = training_examples[4:]

write_jsonl(train_data, "train.jsonl")
write_jsonl(val_data, "val.jsonl")

# Verify the files are valid JSONL
print("\nValidation -- reading back train.jsonl:")
with open("train.jsonl") as f:
    for i, line in enumerate(f):
        obj = json.loads(line)
        roles = [m["role"] for m in obj["messages"]]
        print(f"  Line {i}: roles={roles}, "
              f"assistant_len={len(obj['messages'][-1]['content'])} chars")

---
## Section 4: Training Data Quality Checks

Data quality is the single most important variable in fine-tuning. These checks catch common problems before you spend money on training.

In [ ]:
def validate_fine_tuning_dataset(filepath: str, encoding_name: str = "cl100k_base") -> dict:
    """
    Comprehensive validation of an OpenAI fine-tuning JSONL file.
    Checks format, token counts, duplicates, and consistency.

    Args:
        filepath: Path to the JSONL file
        encoding_name: tiktoken encoding to use for token counting

    Returns:
        Dictionary with validation results and statistics
    """
    enc = tiktoken.get_encoding(encoding_name)
    errors = []
    warnings = []
    token_counts = []
    assistant_token_counts = []
    system_prompts = []
    seen_hashes = set()
    duplicates = 0

    with open(filepath) as f:
        lines = f.readlines()

    if len(lines) == 0:
        return {"valid": False, "errors": ["File is empty"]}

    for i, line in enumerate(lines):
        line_num = i + 1

        # 1. Valid JSON?
        try:
            obj = json.loads(line)
        except json.JSONDecodeError as e:
            errors.append(f"Line {line_num}: Invalid JSON -- {e}")
            continue

        # 2. Has 'messages' key?
        if "messages" not in obj:
            errors.append(f"Line {line_num}: Missing 'messages' key")
            continue

        messages = obj["messages"]

        # 3. Messages is a non-empty list?
        if not isinstance(messages, list) or len(messages) == 0:
            errors.append(f"Line {line_num}: 'messages' must be a non-empty list")
            continue

        # 4. Valid roles?
        valid_roles = {"system", "user", "assistant"}
        for j, msg in enumerate(messages):
            if "role" not in msg or "content" not in msg:
                errors.append(f"Line {line_num}, msg {j}: Missing 'role' or 'content'")
            elif msg["role"] not in valid_roles:
                errors.append(f"Line {line_num}, msg {j}: Invalid role '{msg['role']}'")

        # 5. Must end with assistant message
        if messages[-1].get("role") != "assistant":
            errors.append(f"Line {line_num}: Last message must be from 'assistant'")

        # 6. Must contain at least one user message
        roles = [m.get("role") for m in messages]
        if "user" not in roles:
            errors.append(f"Line {line_num}: Must contain at least one 'user' message")

        # 7. Token count
        total_tokens = sum(len(enc.encode(m.get("content", ""))) for m in messages)
        token_counts.append(total_tokens)

        assistant_tokens = sum(
            len(enc.encode(m.get("content", "")))
            for m in messages if m.get("role") == "assistant"
        )
        assistant_token_counts.append(assistant_tokens)

        if total_tokens > 16385:  # gpt-4o-mini context limit for FT
            warnings.append(f"Line {line_num}: {total_tokens} tokens exceeds 16,385 limit")

        # 8. System prompt consistency
        sys_msgs = [m["content"] for m in messages if m.get("role") == "system"]
        if sys_msgs:
            system_prompts.append(sys_msgs[0])

        # 9. Deduplication check (hash of user + assistant content)
        content_hash = hashlib.md5(
            "".join(m.get("content", "") for m in messages).encode()
        ).hexdigest()
        if content_hash in seen_hashes:
            duplicates += 1
            warnings.append(f"Line {line_num}: Duplicate of a previous example")
        seen_hashes.add(content_hash)

    # 10. System prompt consistency check
    unique_system_prompts = set(system_prompts)
    if len(unique_system_prompts) > 1:
        warnings.append(
            f"Found {len(unique_system_prompts)} different system prompts. "
            f"Consider using a consistent system prompt across all examples."
        )

    # Compile stats
    is_valid = len(errors) == 0
    stats = {}
    if token_counts:
        stats = {
            "num_examples": len(lines),
            "total_tokens": sum(token_counts),
            "avg_tokens_per_example": round(sum(token_counts) / len(token_counts), 1),
            "max_tokens": max(token_counts),
            "min_tokens": min(token_counts),
            "avg_assistant_tokens": round(sum(assistant_token_counts) / len(assistant_token_counts), 1),
            "duplicates": duplicates,
            "unique_system_prompts": len(unique_system_prompts),
        }

    return {
        "valid": is_valid,
        "errors": errors,
        "warnings": warnings,
        "stats": stats,
    }


# Validate our training file
print("Validating train.jsonl...\n")
result = validate_fine_tuning_dataset("train.jsonl")

print(f"Valid: {result['valid']}")
if result["errors"]:
    print(f"\nErrors ({len(result['errors'])}):\n")
    for e in result["errors"]:
        print(f"  {e}")
if result["warnings"]:
    print(f"\nWarnings ({len(result['warnings'])}):\n")
    for w in result["warnings"]:
        print(f"  {w}")
print(f"\nDataset Statistics:")
for k, v in result["stats"].items():
    print(f"  {k}: {v}")

---
## Section 5: Cost Estimation for Fine-Tuning

Fine-tuning costs have two components:
1. **Training cost** -- per token, one-time, paid when the job runs
2. **Inference cost** -- per token at call time (fine-tuned models cost more than base)

The economic logic: training cost amortizes across inference calls. If your fine-tuned model removes 500 tokens of system prompt per call, the savings accumulate rapidly at scale.

In [ ]:
def estimate_fine_tuning_cost(
    num_examples: int,
    avg_tokens_per_example: int,
    n_epochs: int = 3,
    model: str = "gpt-4o-mini-2024-07-18",
    monthly_calls: int = 100_000,
    system_prompt_tokens_saved: int = 500,
    avg_output_tokens: int = 200,
) -> dict:
    """
    Estimate the cost of an OpenAI fine-tuning job and the
    break-even point vs. using the base model with a long prompt.

    Pricing as of early 2025 (check OpenAI pricing page for current rates).
    """
    # Approximate pricing (USD per 1M tokens)
    pricing = {
        "gpt-4o-mini-2024-07-18": {
            "training": 3.00,        # per 1M training tokens
            "input_base": 0.15,      # base model input
            "output_base": 0.60,     # base model output
            "input_ft": 0.30,        # fine-tuned model input
            "output_ft": 1.20,       # fine-tuned model output
        },
        "gpt-4o-2024-08-06": {
            "training": 25.00,
            "input_base": 2.50,
            "output_base": 10.00,
            "input_ft": 3.75,
            "output_ft": 15.00,
        },
    }

    if model not in pricing:
        return {"error": f"Unknown model: {model}. Available: {list(pricing.keys())}"}

    p = pricing[model]

    # Training cost
    total_training_tokens = num_examples * avg_tokens_per_example * n_epochs
    training_cost = (total_training_tokens / 1_000_000) * p["training"]

    # Monthly inference cost: Base model (with long system prompt)
    base_input_tokens_per_call = system_prompt_tokens_saved + 100  # prompt + user msg
    base_monthly_input = monthly_calls * base_input_tokens_per_call
    base_monthly_output = monthly_calls * avg_output_tokens
    base_monthly_cost = (
        (base_monthly_input / 1_000_000) * p["input_base"]
        + (base_monthly_output / 1_000_000) * p["output_base"]
    )

    # Monthly inference cost: Fine-tuned model (no system prompt needed)
    ft_input_tokens_per_call = 100  # just user message
    ft_monthly_input = monthly_calls * ft_input_tokens_per_call
    ft_monthly_output = monthly_calls * avg_output_tokens
    ft_monthly_cost = (
        (ft_monthly_input / 1_000_000) * p["input_ft"]
        + (ft_monthly_output / 1_000_000) * p["output_ft"]
    )

    monthly_savings = base_monthly_cost - ft_monthly_cost
    break_even_months = (
        training_cost / monthly_savings if monthly_savings > 0 else float("inf")
    )

    return {
        "model": model,
        "training": {
            "num_examples": f"{num_examples:,}",
            "avg_tokens_per_example": avg_tokens_per_example,
            "n_epochs": n_epochs,
            "total_training_tokens": f"{total_training_tokens:,}",
            "training_cost": f"${training_cost:.2f}",
        },
        "monthly_inference": {
            "monthly_calls": f"{monthly_calls:,}",
            "base_model_cost": f"${base_monthly_cost:.2f}/mo",
            "fine_tuned_cost": f"${ft_monthly_cost:.2f}/mo",
            "monthly_savings": f"${monthly_savings:.2f}/mo",
        },
        "break_even_months": f"{break_even_months:.1f}" if break_even_months != float("inf") else "Never (FT costs more)",
    }


# Scenario 1: Moderate volume with gpt-4o-mini
print("Scenario 1: Medical note summarizer (moderate volume)")
print("=" * 55)
est = estimate_fine_tuning_cost(
    num_examples=500,
    avg_tokens_per_example=350,
    n_epochs=3,
    model="gpt-4o-mini-2024-07-18",
    monthly_calls=100_000,
    system_prompt_tokens_saved=500,
)
for section, values in est.items():
    if isinstance(values, dict):
        print(f"\n{section}:")
        for k, v in values.items():
            print(f"  {k}: {v}")
    else:
        print(f"\n{section}: {values}")

# Scenario 2: High volume
print("\n\nScenario 2: High-volume customer service (10M calls/month)")
print("=" * 55)
est2 = estimate_fine_tuning_cost(
    num_examples=2000,
    avg_tokens_per_example=400,
    n_epochs=3,
    model="gpt-4o-mini-2024-07-18",
    monthly_calls=10_000_000,
    system_prompt_tokens_saved=800,
)
for section, values in est2.items():
    if isinstance(values, dict):
        print(f"\n{section}:")
        for k, v in values.items():
            print(f"  {k}: {v}")
    else:
        print(f"\n{section}: {values}")

---
## Section 6: OpenAI Fine-Tuning API Walkthrough

The full lifecycle: **upload files -> create job -> monitor progress -> use model**.

OpenAI handles all infrastructure -- GPU allocation, training, model storage, and serving. You bring the data, they handle everything else.

> **Note**: The cells below contain real API calls. Running them will upload files and (optionally) start a fine-tuning job that incurs costs. Review each cell before executing.

In [ ]:
# Step 1: Upload training and validation files

with open("train.jsonl", "rb") as f:
    train_file = client.files.create(file=f, purpose="fine-tune")

with open("val.jsonl", "rb") as f:
    val_file = client.files.create(file=f, purpose="fine-tune")

print(f"Training file uploaded:   {train_file.id}")
print(f"Validation file uploaded: {val_file.id}")
print(f"\nTraining file status:    {train_file.status}")
print(f"Training file bytes:     {train_file.bytes:,}")

In [ ]:
# Step 2: Create the fine-tuning job
# Set DRY_RUN = True to inspect the config without starting a job.

DRY_RUN = True  # Change to False to actually start fine-tuning

job_config = {
    "training_file": train_file.id,
    "validation_file": val_file.id,
    "model": "gpt-4o-mini-2024-07-18",
    "hyperparameters": {
        "n_epochs": "auto",               # OpenAI picks 3-4 for small datasets
        "learning_rate_multiplier": "auto",
        "batch_size": "auto",
    },
    "suffix": "medical-notes-v1",          # appears in the model ID
}

print("Fine-tuning job configuration:")
print(json.dumps(job_config, indent=2))

if DRY_RUN:
    print("\n[DRY RUN] Set DRY_RUN = False to submit this job.")
    job = None
else:
    job = client.fine_tuning.jobs.create(**job_config)
    print(f"\nJob created!")
    print(f"  Job ID: {job.id}")
    print(f"  Status: {job.status}")
    print(f"  Model:  {job.model}")

In [ ]:
# Step 3: Monitor training progress
# Poll job status and stream training events (loss curves).

def monitor_fine_tuning_job(client: OpenAI, job_id: str, poll_interval: int = 60):
    """
    Poll an OpenAI fine-tuning job until completion.
    Prints status updates and training/validation loss.
    """
    print(f"Monitoring job: {job_id}")
    print(f"Polling every {poll_interval} seconds...\n")

    terminal_states = {"succeeded", "failed", "cancelled"}

    while True:
        job = client.fine_tuning.jobs.retrieve(job_id)
        print(f"[{time.strftime('%H:%M:%S')}] Status: {job.status}")

        if job.status in terminal_states:
            break

        # Print recent events (loss values)
        events = client.fine_tuning.jobs.list_events(job_id, limit=5)
        for event in reversed(events.data):
            if "loss" in event.message.lower():
                print(f"  {event.message}")

        time.sleep(poll_interval)

    # Final status
    print(f"\nJob completed with status: {job.status}")
    if job.status == "succeeded":
        print(f"Fine-tuned model ID: {job.fine_tuned_model}")
        print(f"\nUse this model ID in API calls:")
        print(f'  model="{job.fine_tuned_model}"')
    elif job.status == "failed":
        print(f"Error: {job.error}")

    return job


# Uncomment to monitor a real job:
# completed_job = monitor_fine_tuning_job(client, job.id, poll_interval=60)

print("Monitor function defined.")
print("Usage: completed_job = monitor_fine_tuning_job(client, job.id)")

In [ ]:
# Step 4: Use the fine-tuned model
# After training completes, use the model exactly like any other OpenAI model.

# Replace with your actual fine-tuned model ID after training completes:
FINE_TUNED_MODEL = "ft:gpt-4o-mini-2024-07-18:your-org:medical-notes-v1:AbCdEf12"


def query_fine_tuned_model(client: OpenAI, model_id: str, clinical_note: str) -> str:
    """
    Query the fine-tuned model. Note: no system prompt needed!
    The format and behavior are baked into the model weights.
    """
    response = client.chat.completions.create(
        model=model_id,
        messages=[
            {"role": "user", "content": clinical_note}
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content


# Demo call (will fail unless you have a real fine-tuned model)
test_note = (
    "58-year-old male with acute onset left-sided weakness and slurred speech. "
    "CT head shows no hemorrhage. NIHSS score 12. Last known well 90 minutes ago."
)

print("Test clinical note:")
print(f"  {test_note}")
print("\nTo query the fine-tuned model:")
print(f'  result = query_fine_tuned_model(client, "{FINE_TUNED_MODEL}", test_note)')
print("\nKey benefit: No system prompt needed. The format is baked into the weights.")
print("This saves ~500 input tokens per call compared to the prompted base model.")

---
## Section 7: Hyperparameter Selection Guide

Understanding hyperparameters lets you control training quality and avoid common pitfalls.

### OpenAI Fine-Tuning Hyperparameters

| Parameter | Description | Default | Guidelines |
|---|---|---|---|
| `n_epochs` | Passes through training data | `auto` (3-4 for <1K examples) | More epochs = more memorization risk |
| `learning_rate_multiplier` | Scales the base learning rate | `auto` | 0.5-2.0 range for experimentation |
| `batch_size` | Examples per gradient step | `auto` | Larger = more stable, needs more memory |

### LoRA / QLoRA Hyperparameters (for self-hosted fine-tuning)

| Parameter | Typical Range | Notes |
|---|---|---|
| `rank (r)` | 4-64 | 8-16 is the sweet spot for most tasks |
| `alpha` | r to 2r | Controls update magnitude relative to pretrained weights |
| `lora_dropout` | 0.0-0.1 | 0.05 is a common default |
| `learning_rate` | 1e-5 to 5e-4 | Lower than full fine-tuning |
| `target_modules` | all linear layers | Modern best practice; originally just q_proj, v_proj |

In [ ]:
# Hyperparameter recommendation engine

def recommend_hyperparameters(
    num_examples: int,
    task_complexity: str = "medium",  # "low", "medium", "high"
    platform: str = "openai",         # "openai" or "self_hosted"
) -> dict:
    """
    Recommend hyperparameters based on dataset size and task complexity.

    Args:
        num_examples: Number of training examples
        task_complexity: "low" (format change), "medium" (style/domain), "high" (new capability)
        platform: "openai" for hosted API, "self_hosted" for LoRA/QLoRA

    Returns:
        Recommended hyperparameter configuration
    """
    if platform == "openai":
        # Epoch recommendations based on dataset size
        if num_examples < 100:
            n_epochs = 4
            lr_note = "Use auto. Small dataset benefits from more passes."
        elif num_examples < 1000:
            n_epochs = 3
            lr_note = "Use auto. Standard for most use cases."
        elif num_examples < 10000:
            n_epochs = 2
            lr_note = "Consider reducing to 0.8-1.0. Larger datasets converge faster."
        else:
            n_epochs = 1
            lr_note = "Consider reducing to 0.5-0.8. One epoch may suffice."

        return {
            "platform": "OpenAI Fine-Tuning API",
            "n_epochs": n_epochs,
            "learning_rate_multiplier": "auto",
            "batch_size": "auto",
            "lr_note": lr_note,
            "overfitting_risk": "HIGH" if num_examples < 100 else "MEDIUM" if num_examples < 500 else "LOW",
            "recommendation": (
                "Always supply a validation file. Watch for validation loss diverging "
                "from training loss -- that signals overfitting."
            ),
        }
    else:  # self_hosted LoRA/QLoRA
        complexity_config = {
            "low":    {"rank": 8,  "alpha": 16, "lr": 2e-4, "dropout": 0.05},
            "medium": {"rank": 16, "alpha": 32, "lr": 1e-4, "dropout": 0.05},
            "high":   {"rank": 32, "alpha": 64, "lr": 5e-5, "dropout": 0.1},
        }
        cfg = complexity_config[task_complexity]

        if num_examples < 500:
            n_epochs = 5
        elif num_examples < 5000:
            n_epochs = 3
        else:
            n_epochs = 1

        return {
            "platform": "Self-hosted (LoRA/QLoRA)",
            "lora_rank": cfg["rank"],
            "lora_alpha": cfg["alpha"],
            "learning_rate": cfg["lr"],
            "lora_dropout": cfg["dropout"],
            "n_epochs": n_epochs,
            "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                               "gate_proj", "up_proj", "down_proj"],
            "quantization": "4-bit NF4 (QLoRA)" if task_complexity != "high" else "FP16 (LoRA)",
            "optimizer": "paged_adamw_8bit",
            "warmup_ratio": 0.03,
            "gradient_accumulation_steps": 4,
            "catastrophic_forgetting_mitigation": (
                "Include 10-20% general instruction-following data in training mix."
            ),
        }


# Example scenarios
scenarios = [
    ("JSON format enforcement", 200, "low", "openai"),
    ("Medical domain adaptation", 5000, "medium", "self_hosted"),
    ("New language support", 50000, "high", "self_hosted"),
    ("Customer service tone", 1000, "medium", "openai"),
]

for name, n_examples, complexity, platform in scenarios:
    print(f"\n{'='*55}")
    print(f"Task: {name} ({n_examples:,} examples, {complexity} complexity)")
    print(f"{'='*55}")
    rec = recommend_hyperparameters(n_examples, complexity, platform)
    for k, v in rec.items():
        print(f"  {k}: {v}")

---
## Section 8: Evaluation -- Fine-Tuned vs Base Model

Establishing a baseline before fine-tuning is critical. Without it, you cannot know if your fine-tuned model is actually better or just different.

### Evaluation Strategy
1. Define your evaluation set from **real production examples** (not synthetic data)
2. Run the **base model with full prompt** on the evaluation set
3. Run the **fine-tuned model** on the same set
4. Compare on metrics that actually matter: format correctness, factual accuracy, domain vocabulary usage

In [ ]:
# Evaluation framework: compare base model (prompted) vs fine-tuned model

def evaluate_format_correctness(response: str, required_fields: list[str]) -> dict:
    """
    Check if a model response is valid JSON with the required fields.
    This is the primary metric for format-enforcement fine-tuning.
    """
    result = {
        "is_valid_json": False,
        "has_all_fields": False,
        "missing_fields": [],
        "extra_fields": [],
        "field_types_correct": False,
    }

    try:
        parsed = json.loads(response)
        result["is_valid_json"] = True
    except (json.JSONDecodeError, TypeError):
        return result

    # Check required fields
    present_fields = set(parsed.keys())
    required_set = set(required_fields)
    result["missing_fields"] = list(required_set - present_fields)
    result["extra_fields"] = list(present_fields - required_set)
    result["has_all_fields"] = len(result["missing_fields"]) == 0

    # Check field types for our medical note schema
    expected_types = {
        "chief_complaint": str,
        "assessment": str,
        "plan": str,
        "icd10_codes": list,
    }
    types_correct = all(
        isinstance(parsed.get(field), expected_types.get(field, object))
        for field in required_fields
        if field in parsed
    )
    result["field_types_correct"] = types_correct

    return result


def run_evaluation(
    client: OpenAI,
    eval_notes: list[str],
    base_model: str = "gpt-4o-mini",
    fine_tuned_model: str = None,
    system_prompt: str = "",
) -> dict:
    """
    Run evaluation comparing base model (with prompt) vs fine-tuned model.

    Returns comparison metrics.
    """
    required_fields = ["chief_complaint", "assessment", "plan", "icd10_codes"]
    results = {"base": [], "fine_tuned": []}

    for note in eval_notes:
        # Base model: needs full system prompt
        base_response = client.chat.completions.create(
            model=base_model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": note},
            ],
            temperature=0.2,
        )
        base_text = base_response.choices[0].message.content
        base_eval = evaluate_format_correctness(base_text, required_fields)
        base_eval["input_tokens"] = base_response.usage.prompt_tokens
        base_eval["output_tokens"] = base_response.usage.completion_tokens
        results["base"].append(base_eval)

        # Fine-tuned model: no system prompt needed
        if fine_tuned_model:
            ft_response = client.chat.completions.create(
                model=fine_tuned_model,
                messages=[{"role": "user", "content": note}],
                temperature=0.2,
            )
            ft_text = ft_response.choices[0].message.content
            ft_eval = evaluate_format_correctness(ft_text, required_fields)
            ft_eval["input_tokens"] = ft_response.usage.prompt_tokens
            ft_eval["output_tokens"] = ft_response.usage.completion_tokens
            results["fine_tuned"].append(ft_eval)

    # Aggregate metrics
    def aggregate(evals):
        if not evals:
            return {}
        n = len(evals)
        return {
            "json_valid_rate": f"{sum(e['is_valid_json'] for e in evals) / n * 100:.0f}%",
            "all_fields_rate": f"{sum(e['has_all_fields'] for e in evals) / n * 100:.0f}%",
            "types_correct_rate": f"{sum(e['field_types_correct'] for e in evals) / n * 100:.0f}%",
            "avg_input_tokens": round(sum(e.get('input_tokens', 0) for e in evals) / n),
            "avg_output_tokens": round(sum(e.get('output_tokens', 0) for e in evals) / n),
        }

    return {
        "base_model": aggregate(results["base"]),
        "fine_tuned_model": aggregate(results["fine_tuned"]),
        "num_eval_examples": len(eval_notes),
    }


# Run evaluation against the base model only
# (Fine-tuned model evaluation requires a real fine-tuned model ID)
eval_notes = [
    "38-year-old female with 2-day history of dysuria, urinary frequency, and suprapubic pain. "
    "UA positive for leukocyte esterase and nitrites. No fever or flank pain.",

    "67-year-old male with sudden onset of right-sided facial droop and arm weakness 45 minutes ago. "
    "NIHSS 8. CT head negative for hemorrhage. Within tPA window.",

    "22-year-old female presenting with acute asthma exacerbation. Wheezing bilaterally, "
    "SpO2 91% on room air. Using accessory muscles. Peak flow 35% of predicted.",
]

print("Running base model evaluation...\n")
eval_results = run_evaluation(
    client=client,
    eval_notes=eval_notes,
    base_model="gpt-4o-mini",
    system_prompt=SYSTEM_PROMPT,
)

print(f"Evaluation on {eval_results['num_eval_examples']} examples:\n")
print("Base Model (with system prompt):")
for k, v in eval_results["base_model"].items():
    print(f"  {k}: {v}")

if eval_results["fine_tuned_model"]:
    print("\nFine-Tuned Model (no system prompt):")
    for k, v in eval_results["fine_tuned_model"].items():
        print(f"  {k}: {v}")
    print("\nToken savings per call:",
          eval_results['base_model']['avg_input_tokens'] - eval_results['fine_tuned_model']['avg_input_tokens'],
          "input tokens")
else:
    print("\nFine-tuned model: [skipped -- set a real model ID to compare]")

In [ ]:
# LLM-as-Judge evaluation: use a stronger model to score quality

def llm_judge_evaluation(
    client: OpenAI,
    clinical_note: str,
    model_response: str,
    judge_model: str = "gpt-4o",
) -> dict:
    """
    Use a stronger model to evaluate the quality of a fine-tuned model's output.
    Scores on: accuracy, completeness, clinical appropriateness, format adherence.
    """
    judge_prompt = f"""You are evaluating a medical documentation assistant's output.

CLINICAL NOTE:
{clinical_note}

MODEL OUTPUT:
{model_response}

Score the output on these dimensions (1-5 each):
1. accuracy: Are the medical facts correctly extracted?
2. completeness: Are all relevant findings captured?
3. clinical_appropriateness: Does the plan make clinical sense?
4. format_adherence: Is it valid JSON with chief_complaint, assessment, plan, icd10_codes?

Return ONLY a JSON object with these four scores and a brief "reasoning" field.
Example: {{"accuracy": 4, "completeness": 5, "clinical_appropriateness": 4, "format_adherence": 5, "reasoning": "..."}}"""

    response = client.chat.completions.create(
        model=judge_model,
        messages=[{"role": "user", "content": judge_prompt}],
        temperature=0.0,
        response_format={"type": "json_object"},
    )

    return json.loads(response.choices[0].message.content)


# Demo: Judge the base model's output on one example
test_note = eval_notes[0]
base_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": test_note},
    ],
    temperature=0.2,
)
base_output = base_response.choices[0].message.content

print("Clinical note:")
print(f"  {test_note}\n")
print("Base model output:")
print(f"  {base_output}\n")

print("LLM Judge evaluation:")
scores = llm_judge_evaluation(client, test_note, base_output)
for k, v in scores.items():
    print(f"  {k}: {v}")

---
## Section 9: RLHF & DPO -- Alignment Concepts

While OpenAI's hosted fine-tuning handles SFT (supervised fine-tuning), understanding the alignment landscape is important for working with open-weight models.

### RLHF (Reinforcement Learning from Human Feedback)
Three-stage pipeline:
1. **SFT** -- Fine-tune on (prompt, ideal response) pairs
2. **Reward Model** -- Train a model to score response quality based on human rankings
3. **PPO** -- Use RL to optimize the policy toward higher reward, with a KL penalty to stay close to the SFT model

RLHF is what turned GPT-3 (a text completion engine) into ChatGPT (a helpful assistant). It is computationally expensive (3-10x SFT cost) and requires holding multiple models in VRAM simultaneously.

### DPO (Direct Preference Optimization)
DPO achieves the same goal as RLHF in a single training step, without a separate reward model or RL loop. It trains directly on preference pairs (prompt, chosen response, rejected response).

**DPO is now the dominant alignment technique for open-weight models.** It is simpler, more stable, and cheaper than RLHF.

### When to prefer each
| Technique | Best For |
|---|---|
| **SFT** | Format, style, domain adaptation |
| **DPO** | Preference alignment for most teams |
| **RLHF** | Large-scale continuous alignment with online feedback (OpenAI/Anthropic scale) |
| **ORPO** | Combined SFT + alignment in one step for smaller models |

In [ ]:
# DPO dataset format demonstration
# DPO requires (prompt, chosen, rejected) triplets.

dpo_examples = [
    {
        "prompt": "Explain why the sky is blue in one paragraph.",
        "chosen": (
            "The sky appears blue due to a phenomenon called Rayleigh scattering. "
            "When sunlight enters Earth's atmosphere, it collides with gas molecules. "
            "Shorter wavelengths of light (blue and violet) scatter more than longer "
            "wavelengths (red and orange) because scattering intensity is inversely "
            "proportional to the fourth power of wavelength. Although violet light "
            "scatters even more than blue, our eyes are more sensitive to blue light "
            "and the upper atmosphere absorbs some violet, so we perceive the sky as blue."
        ),
        "rejected": (
            "The sky is blue because of the way light interacts with the atmosphere. "
            "Basically, blue light gets scattered around more than other colors. "
            "That's just how it works."
        ),
    },
    {
        "prompt": "What are the health risks of excessive sugar consumption?",
        "chosen": (
            "Excessive sugar consumption is associated with several significant health "
            "risks. Chronically high sugar intake can lead to insulin resistance and "
            "type 2 diabetes, as the pancreas struggles to regulate blood glucose levels. "
            "It contributes to obesity through excess caloric intake and disrupted satiety "
            "signaling. High sugar diets increase the risk of cardiovascular disease by "
            "elevating triglycerides and promoting inflammation. Additionally, sugar is a "
            "primary driver of dental caries and has been linked to non-alcoholic fatty "
            "liver disease. The WHO recommends limiting free sugars to less than 10% of "
            "total daily energy intake."
        ),
        "rejected": (
            "Sugar is really bad for you. It causes diabetes, makes you fat, and rots "
            "your teeth. You should avoid it completely if you want to be healthy. "
            "Sugar is basically poison."
        ),
    },
]

# Write DPO dataset
with open("dpo_train.jsonl", "w") as f:
    for ex in dpo_examples:
        f.write(json.dumps(ex) + "\n")

print(f"DPO dataset: {len(dpo_examples)} preference pairs")
print(f"\nFormat per line:")
print(f"  prompt:   The input question or instruction")
print(f"  chosen:   The preferred (better) response")
print(f"  rejected: The dispreferred (worse) response")
print(f"\nSample:")
print(json.dumps(dpo_examples[0], indent=2)[:400] + "...")

print("\n" + "="*55)
print("To train with DPO using HuggingFace TRL:")
print("""
from trl import DPOTrainer, DPOConfig

dpo_config = DPOConfig(
    beta=0.1,                          # KL coefficient
    learning_rate=5e-7,                # DPO needs very low LR
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
)

trainer = DPOTrainer(
    model=policy_model,
    ref_model=reference_model,
    args=dpo_config,
    train_dataset=dataset,
    processing_class=tokenizer,
)
trainer.train()
""")

---
## Section 10: Synthetic Data Generation for Fine-Tuning

One of the most effective approaches to dataset creation is using a strong LLM (GPT-4o, Claude) to generate training data for fine-tuning a smaller or cheaper model. This is the approach behind Microsoft's Phi models and many open-source fine-tunes.

**Critical warning**: Never use the same synthetic generation process for both training and evaluation data. That contaminates your evaluation.

In [ ]:
# Synthetic data generation pipeline

def generate_synthetic_training_example(
    client: OpenAI,
    seed_topic: str,
    system_prompt: str,
    generator_model: str = "gpt-4o",
) -> dict:
    """
    Use a strong model to generate a training example for fine-tuning a weaker model.

    Args:
        client: OpenAI client
        seed_topic: Topic to generate an example about
        system_prompt: The system prompt the fine-tuned model will use
        generator_model: Strong model to generate with

    Returns:
        Training example in OpenAI fine-tuning format
    """
    meta_prompt = f"""Generate a realistic clinical note and its structured JSON summary.

Topic area: {seed_topic}

The clinical note should be 3-5 sentences describing a patient presentation.
The JSON summary must have exactly these fields:
- chief_complaint (string): one-line summary
- assessment (string): clinical assessment
- plan (string): treatment plan with specific medications and actions
- icd10_codes (list of strings): relevant ICD-10 codes

Return a JSON object with two keys:
- "clinical_note": the input note
- "summary": the structured JSON summary"""

    response = client.chat.completions.create(
        model=generator_model,
        messages=[{"role": "user", "content": meta_prompt}],
        temperature=0.9,  # Higher temp for diversity
        response_format={"type": "json_object"},
    )

    data = json.loads(response.choices[0].message.content)

    # Format as OpenAI fine-tuning example
    return {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": data["clinical_note"]},
            {"role": "assistant", "content": json.dumps(data["summary"])},
        ]
    }


# Generate a few examples
seed_topics = [
    "acute coronary syndrome",
    "diabetic ketoacidosis",
    "community-acquired pneumonia",
]

print("Generating synthetic training examples...\n")
synthetic_examples = []
for topic in seed_topics:
    example = generate_synthetic_training_example(
        client=client,
        seed_topic=topic,
        system_prompt=SYSTEM_PROMPT,
        generator_model="gpt-4o-mini",  # Using mini for demo; use gpt-4o in production
    )
    synthetic_examples.append(example)
    user_msg = example["messages"][1]["content"]
    print(f"Topic: {topic}")
    print(f"  Note: {user_msg[:100]}...")
    print()

print(f"Generated {len(synthetic_examples)} synthetic training examples.")
print("\nIn production, you would:")
print("  1. Generate 500-5,000 examples across diverse topics")
print("  2. Filter for quality (LLM-as-judge or manual review)")
print("  3. Deduplicate")
print("  4. Mix with any available real examples")
print("  5. Hold out REAL (non-synthetic) examples for evaluation")

---
## Summary

In this notebook we covered the full fine-tuning lifecycle:

1. **Decision framework** -- When to fine-tune vs. prompt engineer vs. RAG (behavior problems are where fine-tuning shines)
2. **LoRA & QLoRA** -- Parameter-efficient fine-tuning with memory calculations showing how QLoRA makes 70B model tuning feasible on a single GPU
3. **Data preparation** -- JSONL format for OpenAI, with quality checks for format validation, deduplication, and token counting
4. **OpenAI Fine-Tuning API** -- Full walkthrough: upload, create job, monitor loss curves, use the model
5. **Hyperparameter selection** -- Recommendations based on dataset size and task complexity
6. **Evaluation** -- Format correctness metrics and LLM-as-judge scoring to compare base vs. fine-tuned models
7. **Cost estimation** -- Training cost amortization and break-even analysis
8. **Alignment techniques** -- RLHF and DPO concepts with dataset format examples
9. **Synthetic data generation** -- Using strong models to create training data for weaker models

### Key Takeaways
- Fine-tuning is for **behavior problems**, not knowledge problems (use RAG for knowledge)
- **Data quality >> data quantity** -- 100 perfect examples beat 10,000 mediocre ones
- **Always establish a baseline** before fine-tuning and evaluate on real production data
- LoRA/QLoRA make fine-tuning accessible on consumer hardware
- OpenAI hosted fine-tuning eliminates infrastructure overhead but locks you into their platform

**Next module**: `05-llm-hosting.html` -- Deploying and serving LLMs in production